In [ ]:
import pandas as pd
df = pd.read_csv('combined_data.csv')
df

In [ ]:
df=df.drop(columns=['html'])
df=df.drop(columns=['Unnamed: 0'])
df

In [ ]:
df['M1']=df['in_degree']-df['out_degree']

In [ ]:
train_data=df[['frequency_num','Time gap','M1','eigenvector_centrality','clustering_coefficient','degree_centrality','betweenness_centrality','closeness','construct_label','dynamic_data']]
train_data.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
# 对类别数据进行独热编码
df_encoded = pd.get_dummies(train_data, columns=['dynamic_data'])

# 选择连续数据特征
continuous_features = ['frequency_num','Time gap','M1','eigenvector_centrality','clustering_coefficient','degree_centrality','betweenness_centrality','closeness','construct_label']

# 标准化连续数据
scaler = StandardScaler()
df_encoded[continuous_features] = scaler.fit_transform(df_encoded[continuous_features])

df_encoded.head()

In [ ]:
import pandas as pd
import numpy as np

# 计算每一列的熵
def entropy(series):
    p = series / series.sum()
    return -np.sum(p * np.log(p))

In [ ]:
def entropy_data(df):
    entropies = df.apply(entropy)

    # 计算每一列的权重
    weights = 1 - entropies / entropies.sum()

    # 将权重应用于数据并合并为一列
    weighted_sum = (df * weights).sum(axis=1)

    # 添加合并后的列到DataFrame
    df['Merged'] = weighted_sum

    return(df)

In [ ]:
M2=df_encoded[['eigenvector_centrality','clustering_coefficient','degree_centrality']]
M3=df_encoded[['betweenness_centrality','closeness','construct_label']]

In [ ]:
M2=entropy_data(M2)
M3=entropy_data(M3)

In [ ]:
after_M2=M2[['Merged']]
after_M3=M3[['Merged']]

In [ ]:
temp_data=pd.concat([df_encoded[['M1']],after_M2,after_M3],axis=1)
temp_data.columns=['M1','M2','M3']
temp_data.head()

In [ ]:
temp_data=entropy_data(temp_data)
temp_data.head()

In [ ]:
last_data=pd.DataFrame()
last_data=pd.concat([df_encoded[['frequency_num','Time gap']],temp_data[['Merged']],df_encoded[['dynamic_data_增长','dynamic_data_稳定','dynamic_data_衰退']]], axis=1)
last_data.columns=['F','R','M','T1','T2','T3']
last_data

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score,davies_bouldin_score

sse=[]
CH_data=[]
DB_data=[]

for i in range(2,20):
    kmeans = KMeans(n_clusters=i,init='k-means++',max_iter=300,n_init=10,random_state=0)
    result_list =kmeans.fit_predict(last_data)
    
    CH_index=calinski_harabasz_score(last_data,result_list)
    DB_index=davies_bouldin_score(last_data,result_list)
    
    print(f"方差比为: {CH_index}")
    print(f"DB值为: {DB_index}")
    print('SSE值为:',kmeans.inertia_)
    
    sse.append(kmeans.inertia_)
    CH_data.append(CH_index)
    DB_data.append(DB_index)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(2,20),sse)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
plt.plot(range(2,20),CH_data)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('CH_data')
plt.show()

In [ ]:
CH_data

In [ ]:
plt.plot(range(2,20),DB_data)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('DB_data')
plt.show()

In [ ]:
DB_data

## 最终选择聚类个数为6，此时SSE处于拐点位置，方差比最大，DB值较小 

In [ ]:
kmeans_model = KMeans(n_clusters=6,init='k-means++',max_iter=300,n_init=10,random_state=0)
kmeans_model.fit(last_data)

In [ ]:
kmeans_cc=kmeans_model.cluster_centers_   # 聚类中心
kmeans_cc

In [ ]:
temp_data=pd.DataFrame(kmeans_cc)
temp_data.columns=['F','R','M','T1','T2','T3']

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 初始化 MinMaxScaler
scaler = MinMaxScaler()

# 对 DataFrame 中的每列数据进行归一化
df_normalized = pd.DataFrame(scaler.fit_transform(temp_data), columns=temp_data.columns)

df_normalized

In [ ]:
columns_order = ['F','M','R','T1','T2','T3']  # 这里是你想要的新的列顺序

In [ ]:
# 使用 loc 方法重新排列列
df_normalized=df_normalized.loc[:, columns_order]

In [ ]:
from pyecharts.charts import Radar
from pyecharts import options as opts
import numpy as np

#客户价值雷达图


radar = Radar()
radar.add_schema(
                 textstyle_opts=opts.TextStyleOpts(color='rgb(238, 197, 102)'),
                 axisline_opt=opts.LineStyleOpts(is_show=True, color='rgba(238, 197, 102, 1)'),
                 schema=[opts.RadarIndicatorItem(name='知识共享频率',min_=0, max_=1),
                         opts.RadarIndicatorItem(name="M",min_=0, max_=1),
                         opts.RadarIndicatorItem(name="最近知识共享距离的时间",min_=0, max_=1),
                         opts.RadarIndicatorItem(name="T1",min_=0, max_=1),
                         opts.RadarIndicatorItem(name="T2",min_=0, max_=1),
                         opts.RadarIndicatorItem(name="T3",min_=0, max_=1),
                         ])

radar.add('群体一', [list(df_normalized.iloc[0])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#000000', width=0.1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#000000', opacity=0.1))

radar.add('群体二', [list(df_normalized.iloc[1])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#5cb047', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#5cb047', opacity=0.05))

radar.add('群体三', [list(df_normalized.iloc[2])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#e1306c', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#e1306c',opacity=0.05))

radar.add('群体四', [list(df_normalized.iloc[3])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#f77737', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#f77737',opacity=0.05))

radar.add('群体五', [list(df_normalized.iloc[4])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#f77737', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#f77737',opacity=0.05))

radar.add('群体六', [list(df_normalized.iloc[5])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#f77737', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#f77737',opacity=0.05))

radar.set_global_opts(legend_opts=opts.LegendOpts(is_show=True, selected_mode='flase', pos_bottom=5),
                      title_opts=opts.TitleOpts(title="客户价值雷达图", pos_left='center',
                                                title_textstyle_opts=opts.TextStyleOpts(font_size=20)))


radar.render_notebook()
